# Chapter 15 — Application architecture: MVC/MVP, services, repositories, and navigation

**Book role:** Application design  
**Suggested book space:** about 10 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Separate widgets, business rules, and persistence code.
- Choose practical controller/service/repository responsibilities.
- Pass plain data structures across boundaries.
- Refactor a callback-heavy prototype into testable modules.

## 1. Why one giant callback does not scale

A prototype often starts with a button callback that reads fields, loads a file, cleans columns, executes a query, calculates metrics, updates five widgets, and writes an export. That callback quickly becomes difficult to test and change. The solution is not necessarily a textbook Model–View–Controller framework; it is a clear boundary between responsibilities.

A **view** builds widgets and renders state. A **controller** interprets user actions and chooses the next operation. A **service** owns a use case such as “summarize current sales” or “validate imported transactions.” A **repository/adapter** knows how to read and write CSV, Excel, MySQL, or MongoDB. Business rules should not require a running Tk window, and database queries should not depend on the layout of a form.

### Worked example 1: Use a service function without Tkinter

The service accepts plain inputs and returns data; it never calls `.get()` on a widget. This boundary also makes it straightforward to unit-test.

In [ ]:
from dataclasses import dataclass
import pandas as pd

@dataclass(frozen=True)
class SalesCriteria:
    region: str | None = None
    category: str | None = None
    include_cancelled: bool = False


def filter_sales(frame: pd.DataFrame, criteria: SalesCriteria) -> pd.DataFrame:
    result = frame.copy()
    if not criteria.include_cancelled and "status" in result.columns:
        result = result[result["status"].ne("Cancelled")]
    if criteria.region and criteria.region != "All":
        result = result[result["region"].eq(criteria.region)]
    if criteria.category and criteria.category != "All":
        result = result[result["category"].eq(criteria.category)]
    return result

# This service can be tested in a regular Python process.
sample = pd.DataFrame({"region": ["North", "South"], "category": ["Tech", "Food"], "status": ["Paid", "Cancelled"]})
print(filter_sales(sample, SalesCriteria(region="North")))

## 2. Define service contracts around plain Python values

A service method might accept a `FilterCriteria` dataclass and return a `SummaryResult` containing row count, sales, profit, and a DataFrame. It should not reach into Tk variables or set label text. This allows the same service to be used by a notebook, a command-line job, a test, and the GUI. Data contracts also make errors easier to interpret: invalid criteria should be different from a database outage or a missing file.

Use exceptions at boundaries deliberately. A repository can raise a specific exception or return a typed result; a controller catches expected failures and maps them to user-facing messages while the logger preserves context. Avoid catching every exception and ignoring it; silent failure makes a dashboard untrustworthy.

### Worked example 2: A thin controller callback

This is a design sketch, not a standalone cell: it belongs inside a controller with `view`, state variables, and a DataFrame. The important point is that the callback coordinates operations rather than implementing every rule itself.

In [ ]:
def on_apply_filters(self):
    """Illustrative method inside a controller class."""
    criteria = SalesCriteria(
        region=self.region_var.get(),
        category=self.category_var.get(),
        include_cancelled=self.include_cancelled_var.get(),
    )
    try:
        filtered = filter_sales(self.sales_df, criteria)
    except (KeyError, ValueError) as exc:
        self.show_error(f"Could not apply filters: {exc}")
        return
    self.view.render_records(filtered)
    self.view.set_status(f"Showing {len(filtered):,} matching rows")

## 3. Dependency direction and future changes

The view should depend on a service interface, and the service should depend on repository behavior—not the reverse. The UI should not know whether records came from Excel, MySQL, or MongoDB. An adapter can normalize the source into a canonical DataFrame schema, after which analysis logic behaves consistently.

This is especially valuable for data analytics because storage and analysis tools change independently. A student can prototype with CSV, later connect to a database, and preserve the metrics and visualization code if the contract stays stable. Avoid premature abstraction, though: introduce an interface where there are multiple concrete sources, testability needs, or a genuine reason to isolate dependencies.

## Guided lab: Lab: refactor a monolithic callback

Choose a callback from an earlier example and divide it into input parsing/validation, service calculation, repository access, and UI rendering. Write a short module dependency diagram and one unit test for the service. The service test must run without creating a Tk root or connecting to a database.

## Review questions and practice

1. What is the difference between a view and a service?
2. Why should storage details be hidden behind a repository or adapter?
3. What data types are convenient across layers?
4. Why is a service easier to test without widgets?

## Further reading

- Python dataclasses: https://docs.python.org/3/library/dataclasses.html
- Python modules/packages: https://docs.python.org/3/tutorial/modules.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.